# Lab 2: Edges - Parallel Execution
LangGraph makes it incredibly easy to execute tasks in parallel. If a node points to multiple downstream nodes, those nodes will execute simultaneously. When they complete, their results are combined back into the state.

To merge state updates from parallel nodes, we use `Annotated` with a reducer function (like `operator.add`). This prevents parallel nodes from overwriting each other's updates.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### Define Parallel State and Nodes

In [2]:
from typing import TypedDict, List, Annotated
import operator
from langgraph.graph import StateGraph, START, END

# Define state with a reducer to append values instead of replacing them
class ParallelState(TypedDict):
    input_number: int
    # Annotating with operator.add means state updates to 'values' will be appended to the list
    values: Annotated[List[str], operator.add]

# Define Nodes
def start_node(state: ParallelState):
    print("--- Executing start_node ---")
    return {"values": ["Start"]}

def double_node(state: ParallelState):
    print("--- Executing double_node (parallel branch 1) ---")
    result = state["input_number"] * 2
    return {"values": [f"Doubled: {result}"]}

def square_node(state: ParallelState):
    print("--- Executing square_node (parallel branch 2) ---")
    result = state["input_number"] ** 2
    return {"values": [f"Squared: {result}"]}

def join_node(state: ParallelState):
    print("--- Executing join_node ---")
    return {"values": ["End of Workflow"]}

# Build Graph
parallel_builder = StateGraph(ParallelState)
parallel_builder.add_node("start", start_node)
parallel_builder.add_node("double", double_node)
parallel_builder.add_node("square", square_node)
parallel_builder.add_node("join", join_node)

# Configure Parallel Paths
parallel_builder.add_edge(START, "start")

# Branch from 'start' to BOTH 'double' and 'square' in parallel
parallel_builder.add_edge("start", "double")
parallel_builder.add_edge("start", "square")

# Merge both paths into 'join'
parallel_builder.add_edge("double", "join")
parallel_builder.add_edge("square", "join")

parallel_builder.add_edge("join", END)

parallel_graph = parallel_builder.compile()

try:
    parallel_graph.get_graph().print_ascii()
except:
    pass

        +-----------+        
        | __start__ |        
        +-----------+        
              *              
              *              
              *              
          +-------+          
          | start |          
          +-------+          
         *         *         
       **           **       
      *               *      
+--------+        +--------+ 
| double |        | square | 
+--------+        +--------+ 
         *         *         
          **     **          
            *   *            
          +------+           
          | join |           
          +------+           
              *              
              *              
              *              
        +---------+          
        | __end__ |          
        +---------+          


### Execute Parallel Graph

In [3]:
result_state = parallel_graph.invoke({"input_number": 6, "values": []})
print("\n--- Final State Values ---")
for val in result_state["values"]:
    print("- ", val)

--- Executing start_node ---
--- Executing double_node (parallel branch 1) ---
--- Executing square_node (parallel branch 2) ---
--- Executing join_node ---

--- Final State Values ---
-  Start
-  Doubled: 12
-  Squared: 36
-  End of Workflow
